In [11]:
import xarray as xr
import pandas as pd

In [12]:
def select_window(ds, date, days_before=1, days_after=3):
    """
    Selecciona una ventana temporal alrededor de una fecha.

    Parameters
    ----------
    ds : xarray.Dataset o xarray.DataArray
    date : str o datetime-like
        Fecha central (D0).
    days_before : int
        Número de días antes de D0.
    days_after : int
        Número de días después de D0.

    Returns
    -------
    xarray.Dataset o xarray.DataArray
    """
    
    date = pd.Timestamp(date)

    start = date - pd.Timedelta(days=days_before)
    end   = date + pd.Timedelta(days=days_after)

    return ds.sel(time=slice(start, end))


def select_window_ar(ds, date, days_before=1, days_after=3):
    """
    Selecciona una ventana temporal alrededor de una fecha
    y conserva únicamente los registros de 00 UTC.
    """

    date = pd.Timestamp(date)

    start = date - pd.Timedelta(days=days_before)
    end = date + pd.Timedelta(days=days_after)

    ds_sel = ds.sel(time=slice(start, end))

    # Mantener solo 00 UTC
    ds_sel = ds_sel.sel(time=ds_sel.time.dt.hour == 0)

    return ds_sel

In [13]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import numpy as np
import re
import os

# =========================================================
# === 1. CONFIGURACIÓN BASE DEL MAPA
# =========================================================
def setup_map(ax, proj_map, proj_data):
    """Configura el mapa base en un eje Cartopy."""
    ax.coastlines(resolution='50m')
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.LAND, facecolor='lightgray')


# =========================================================
# === 2. PLOT T2M + Z500
# =========================================================
def plot_temp_z500(ax, lon, lat, temp_field, z500_field,
                   levels_temp, levels_z500,
                   proj_data):

    # --- Temperature filled contours ---
    temp_plot = ax.contourf(
        lon, lat, temp_field,
        levels=levels_temp,
        cmap='bwr',
        extend='both',
        transform=proj_data
    )

    # --- Z500 gray contour lines ---
    c = ax.contour(
        lon, lat, z500_field,
        levels=levels_z500,
        colors='gray',
        linewidths=1.0,
        transform=proj_data
    )

    ax.clabel(
        c,
        fmt='%d',
        fontsize=7,
        colors='gray'
    )

    return temp_plot

# =========================================================
# === 3. PLOT PRECIPITACIÓN
# =========================================================
def plot_precip(ax, lon, lat, precip_field,
                levels_precip, cmap_precip,
                proj_data):

    precip_plot = ax.contourf(
        lon,
        lat,
        precip_field,
        levels=levels_precip,
        cmap=cmap_precip,
        extend='max',
        transform=proj_data
    )

    return precip_plot

def plot_shapefile_mask(ax, lon, lat, mask, color='magenta', lw=2, proj_data=ccrs.PlateCarree()):
    """Plotea un shapefile o mascarilla como contorno."""
    ax.contour(lon, lat, mask, levels=[0.5], colors=color,
               linewidths=lw, transform=proj_data)


In [14]:
from matplotlib.colors import ListedColormap

colors = [
    "#f0f0f0",
    "#7ea6ff",
    "#3366ff",
    "#1e90ff",
    "#00c5cd",
    "#00c878",
    "#32cd32",
    "#9acd32",
    "#ffd54f",
    "#ffb347",
    "#ff7f50",
    "#ff4500",
    "#ff1493",
    "#c050ff",
    "#6a3d9a"
]

cmap_precip = ListedColormap(colors)


In [15]:
filename = '~/ERA5/T2max_daily_anomalies.nc'
t2m = xr.open_dataset(filename)

filename = '~/ERA5/precip_daily_totals.nc'
rrr = xr.open_dataset(filename)

filename = '~//ERA5/Z500_daily_anomalies_75.nc'
z500 = (xr.open_dataset(filename)/9.98).squeeze()


In [16]:
filename = '/home/cr2/cmtorres/ARs_DATA/Global_AR_dataset_Guan.nc'
AR_shape = xr.open_dataset(filename).sel(time=slice('1980-01-01', '2022-12-31'), ens = 1, lev = 0)
AR_shape.coords['lon'] = (AR_shape.coords['lon'] + 180) % 360 - 180
AR_shape = AR_shape.sortby(AR_shape.lon)


In [17]:
filename = '/home/cr2/cmtorres/ARs_DATA/AR_Buffet_1979_2023.nc'
AR_shape_B = xr.open_dataset(filename)
AR_shape_B.coords['lon'] = (AR_shape_B.coords['lon'] + 180) % 360 - 180
AR_shape_B = AR_shape_B.sortby(AR_shape_B.lon)

In [18]:
df_sel = pd.read_csv('/home/cr2/cmtorres/repos/Patagonia-Antarctic_Peninsula/data/top10_sel.csv')
df_sel

,rank,start_date,end_date,n_days,temperature,precipitation,compound_index
0,1,2008-01-25,2008-01-25,1,10.377229,23.679468,4.562696
1,2,2021-02-09,2021-02-11,3,10.445299,23.127605,4.470847
2,3,1988-03-08,1988-03-08,1,6.378127,31.709547,4.137589
3,4,2015-03-24,2015-03-26,2,9.311523,20.376303,3.150394
4,5,1984-12-26,1984-12-27,2,10.067975,17.973000,3.019746
5,6,2019-02-04,2019-02-04,1,12.865652,10.217766,2.806996
6,7,2011-12-16,2011-12-16,1,8.367545,20.919992,2.727437
7,8,1982-03-02,1982-03-03,2,7.754361,21.698622,2.554234
8,9,1990-02-10,1990-02-12,2,8.149093,19.921423,2.361217
9,10,2013-03-03,2013-03-04,2,8.168554,19.437796,2.257191


In [19]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

number_case = 1

# =========================================================
# SELECCIONAR VARIABLES
# =========================================================
for case in range(6):

    date_sel = df_sel['start_date'][case]

    t2m_sel = select_window(t2m, date_sel).tmax_anomaly
    z500_sel = select_window(z500, date_sel).z_anomaly
    rrr_sel = select_window(rrr, date_sel).precip_daily

    AR_shape_12h_sel = select_window_ar(AR_shape, date_sel).shapemap
    AR_shape_12h_B_sel = select_window_ar(AR_shape_B, date_sel).ar_binary_tag

    lon = t2m_sel.longitude
    lat = t2m_sel.latitude

    # =========================================================
    # NIVELES
    # =========================================================

    levels_temp = np.arange(-12, 13, 1)

    levels_z500 = np.arange(-300, 301, 50)
    levels_z500 = levels_z500[levels_z500 != 0]

    levels_precip = np.arange(0, 15, 1.0)

    # =========================================================
    # PROYECCIONES
    # =========================================================

    proj_map = ccrs.Orthographic(-70, -60)
    proj_data = ccrs.PlateCarree()

    # =========================================================
    # FIGURA
    # =========================================================

    fig = plt.figure(figsize=(12, 14))

    gs = gridspec.GridSpec(
        4,
        3,
        figure=fig,
        wspace=0.08,
        hspace=0.12
    )

    # =========================================================
    # TEMPERATURA + Z500
    # =========================================================

    for t in range(5):

        row = t // 3
        col = t % 3

        ax = fig.add_subplot(
            gs[row, col],
            projection=proj_map
        )

        setup_map(ax, proj_map, proj_data)

        temp_plot = plot_temp_z500(
            ax,
            lon,
            lat,
            t2m_sel.isel(time=t),
            z500_sel.isel(time=t),
            levels_temp,
            levels_z500,
            proj_data
        )

        date_str = np.datetime_as_string(
            t2m_sel.time.isel(time=t).values,
            unit='D'
        )

        ax.set_title(
            date_str,
            fontsize=11
        )


    # =========================================================
    # COLORBAR EN SEXTO PANEL
    # =========================================================

    cax_parent = fig.add_subplot(gs[1, 2])
    cax_parent.axis('off')

    cax = inset_axes(
        cax_parent,
        width="90%",   # ancho de la colorbar
        height="20%",  # altura
        loc='center'
    )

    cb = plt.colorbar(
        temp_plot,
        cax=cax,
        orientation='horizontal'
    )

    cb.set_label(
        'Air temperature anomaly (°C)',
        fontsize=12
    )

    # =========================================================
    # PRECIPITACIÓN
    # =========================================================

    for t in range(5):

        row = 2 + t // 3
        col = t % 3

        ax = fig.add_subplot(
            gs[row, col],
            projection=proj_map
        )

        setup_map(ax, proj_map, proj_data)

        precip_plot = plot_precip(
            ax,
            lon,
            lat,
            rrr_sel.isel(time=t),
            levels_precip,
            cmap_precip,
            proj_data
        )

        date_str = np.datetime_as_string(
            rrr_sel.time.isel(time=t).values,
            unit='D'
        )

        ax.set_title(
            date_str,
            fontsize=11
        )

        mask1 = AR_shape_12h_sel.isel(time=t).isnull().astype(int)
        plot_shapefile_mask(ax, AR_shape_12h_sel.lon, AR_shape_12h_sel.lat, mask1, color='magenta', lw=1.)

        mask2 = AR_shape_12h_B_sel.isel(time=t)
        plot_shapefile_mask(ax, AR_shape_12h_B_sel.lon, AR_shape_12h_B_sel.lat, mask2, color='lime', lw=1.)

    # =========================================================
    # COLORBAR PRECIPITACIÓN
    # =========================================================

    cax_parent1 = fig.add_subplot(gs[3, 2])
    cax_parent1.axis('off')

    cax1 = inset_axes(
        cax_parent1,
        width="90%",   # ancho de la colorbar
        height="20%",  # altura
        loc='center'
    )


    cb_precip = plt.colorbar(
        precip_plot,
        cax=cax1,
        orientation='horizontal'
    )

    cb_precip.set_label(
        r'Total precipitation (mmWE day$^{-1}$)',
        fontsize=11
    )

    fig.savefig(f"fig/case_{number_case}.png", dpi=300,
                facecolor='w', bbox_inches='tight')

    number_case += 1

    plt.close(fig)

In [21]:
dates_sel = ['2020-02-10', '2022-02-07']

cases = [69, 53]

for number_case, (case, date_sel) in enumerate(
        zip(cases, dates_sel), start=1):

    print(f'Processing case {case} ({date_sel})')
    
    t2m_sel = select_window(t2m, date_sel).tmax_anomaly
    z500_sel = select_window(z500, date_sel).z_anomaly
    rrr_sel = select_window(rrr, date_sel).precip_daily

    AR_shape_12h_sel = select_window_ar(AR_shape, date_sel).shapemap
    AR_shape_12h_B_sel = select_window_ar(AR_shape_B, date_sel).ar_binary_tag

    lon = t2m_sel.longitude
    lat = t2m_sel.latitude

    # =========================================================
    # NIVELES
    # =========================================================

    levels_temp = np.arange(-12, 13, 1)

    levels_z500 = np.arange(-300, 301, 50)
    levels_z500 = levels_z500[levels_z500 != 0]

    levels_precip = np.arange(0, 15, 1.0)

    # =========================================================
    # PROYECCIONES
    # =========================================================

    proj_map = ccrs.Orthographic(-70, -60)
    proj_data = ccrs.PlateCarree()

    # =========================================================
    # FIGURA
    # =========================================================

    fig = plt.figure(figsize=(12, 14))

    gs = gridspec.GridSpec(
        4,
        3,
        figure=fig,
        wspace=0.08,
        hspace=0.12
    )

    # =========================================================
    # TEMPERATURA + Z500
    # =========================================================

    for t in range(5):

        row = t // 3
        col = t % 3

        ax = fig.add_subplot(
            gs[row, col],
            projection=proj_map
        )

        setup_map(ax, proj_map, proj_data)

        temp_plot = plot_temp_z500(
            ax,
            lon,
            lat,
            t2m_sel.isel(time=t),
            z500_sel.isel(time=t),
            levels_temp,
            levels_z500,
            proj_data
        )

        date_str = np.datetime_as_string(
            t2m_sel.time.isel(time=t).values,
            unit='D'
        )

        ax.set_title(
            date_str,
            fontsize=11
        )


    # =========================================================
    # COLORBAR EN SEXTO PANEL
    # =========================================================

    cax_parent = fig.add_subplot(gs[1, 2])
    cax_parent.axis('off')

    cax = inset_axes(
        cax_parent,
        width="90%",   # ancho de la colorbar
        height="20%",  # altura
        loc='center'
    )

    cb = plt.colorbar(
        temp_plot,
        cax=cax,
        orientation='horizontal'
    )

    cb.set_label(
        'Air temperature anomaly (°C)',
        fontsize=12
    )

    # =========================================================
    # PRECIPITACIÓN
    # =========================================================

    for t in range(5):

        row = 2 + t // 3
        col = t % 3

        ax = fig.add_subplot(
            gs[row, col],
            projection=proj_map
        )

        setup_map(ax, proj_map, proj_data)

        precip_plot = plot_precip(
            ax,
            lon,
            lat,
            rrr_sel.isel(time=t),
            levels_precip,
            cmap_precip,
            proj_data
        )

        date_str = np.datetime_as_string(
            rrr_sel.time.isel(time=t).values,
            unit='D'
        )

        ax.set_title(
            date_str,
            fontsize=11
        )

        mask1 = AR_shape_12h_sel.isel(time=t).isnull().astype(int)
        plot_shapefile_mask(ax, AR_shape_12h_sel.lon, AR_shape_12h_sel.lat, mask1, color='magenta', lw=1.)

        mask2 = AR_shape_12h_B_sel.isel(time=t)
        plot_shapefile_mask(ax, AR_shape_12h_B_sel.lon, AR_shape_12h_B_sel.lat, mask2, color='lime', lw=1.)

    # =========================================================
    # COLORBAR PRECIPITACIÓN
    # =========================================================

    cax_parent1 = fig.add_subplot(gs[3, 2])
    cax_parent1.axis('off')

    cax1 = inset_axes(
        cax_parent1,
        width="90%",   # ancho de la colorbar
        height="20%",  # altura
        loc='center'
    )


    cb_precip = plt.colorbar(
        precip_plot,
        cax=cax1,
        orientation='horizontal'
    )

    cb_precip.set_label(
        r'Total precipitation (mmWE day$^{-1}$)',
        fontsize=11
    )

    fig.savefig(f"fig/case_{case}.png", dpi=300,
                facecolor='w', bbox_inches='tight')


    plt.close(fig)

Processing case 69 (2020-02-10)


Processing case 53 (2022-02-07)
